# 17 — Compare Original, Full-Retrained, and Unlearned Profile-Memory Models

**Purpose:** compare the three profile-memory models with one fixed evaluation plan.

The primary result is exact recall on the six fields deliberately reinforced in Notebook 14. F1, AUROC, and PR-AUC measure whether correct-answer confidence can distinguish trained-profile recipients from unseen controls.

## Pipeline

1. Load the same forget, retain, and control groups.
2. Load original, full-retrained, and targeted-unlearned models.
3. Ask the same six focused questions for every group.
4. Compare exact recall, membership metrics, and runtime.

In [ ]:
%pip install -q -U unsloth trl datasets scikit-learn

from pathlib import Path
import json
import sys
import pandas as pd
import torch

REPO_OVERRIDE = None
repo_candidates = [Path('/content/qub-machine-unlearning'), Path.cwd(), Path.cwd().parent]
if REPO_OVERRIDE:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))
REPO_ROOT = next((path for path in repo_candidates if (path / 'code' / 'final_submission').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Clone the repository into /content, then rerun this cell.')
sys.path.insert(0, str(REPO_ROOT / 'code' / 'final_submission' / 'notebooks'))
from profile_memory_unlearning_common import *

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required.')
set_seed()
print('GPU:', torch.cuda.get_device_name(0))
print('Repository:', REPO_ROOT)

In [ ]:
PROFILES = load_profiles(REPO_ROOT)
CONTRACT = load_contract(REPO_ROOT)
PATHS = paths(REPO_ROOT)

MODEL_DIRS = {
    'original': Path('/content/qwen_profile_memory_model'),
    'full_retrained': PATHS['artifacts'] / 'full_retrained_profile_memory_without_forget',
    'targeted_unlearned': PATHS['artifacts'] / 'targeted_unlearned_profile_memory_model',
}
PLAN = evaluation_plan(PROFILES, CONTRACT)
assert len(PLAN) == 3_600
display(PLAN.groupby('group').size().to_frame('questions'))

## Evaluate every model with the same questions

The original model selects one confidence threshold from retain recipients versus controls. That threshold is then reused unchanged for full retraining and targeted unlearning.

In [ ]:
recall_rows, membership_rows = [], []
# Calibrate once on the original model's retained recipients versus controls.
# Every model and group then uses this same threshold, so F1 is comparable.
original_model, original_tokenizer = load_adapter(MODEL_DIRS['original'])
original_calibration_rows = evaluate(
    original_model, original_tokenizer, PLAN,
    PATHS['results'] / 'original_focused_profile_rows.csv',
)
_, FIXED_THRESHOLD = membership_metrics(original_calibration_rows, 'retain')
(PATHS['results'] / 'fixed_profile_membership_threshold.json').write_text(
    json.dumps({'threshold': FIXED_THRESHOLD, 'source': 'original retain versus control calibration'}, indent=2),
    encoding='utf-8',
)
del original_model
torch.cuda.empty_cache()

for name, model_dir in MODEL_DIRS.items():
    model, tokenizer = load_adapter(model_dir)
    rows = evaluate(
        model, tokenizer, PLAN,
        PATHS['results'] / f'{name}_focused_profile_rows.csv',
    )
    summary = recall_summary(rows)
    summary['model'] = name
    recall_rows.append(summary)

    for positive_group in ['forget', 'retain']:
        metric, _ = membership_metrics(
            rows,
            positive_group,
            threshold=FIXED_THRESHOLD,
        )
        metric['model'] = name
        membership_rows.append(metric)
    del model
    torch.cuda.empty_cache()

recall_comparison = pd.concat(recall_rows, ignore_index=True)
membership_comparison = pd.DataFrame(membership_rows)
recall_comparison.to_csv(PATHS['results'] / 'final_profile_recall_comparison.csv', index=False)
membership_comparison.to_csv(PATHS['results'] / 'final_profile_membership_comparison.csv', index=False)
display(recall_comparison)
display(membership_comparison)

## Runtime comparison

In [ ]:
full_retraining_runtime = pd.read_csv(PATHS['results'] / 'full_retraining_runtime.csv')
unlearning_runtime = pd.read_csv(PATHS['results'] / 'targeted_unlearning_runtime.csv')
full_retraining_seconds = float(full_retraining_runtime['seconds'].sum())
unlearning_seconds = float(unlearning_runtime['seconds'].sum())
runtime_comparison = pd.DataFrame([
    {'method': 'full_retraining', 'seconds': full_retraining_seconds, 'minutes': full_retraining_seconds / 60},
    {'method': 'targeted_unlearning', 'seconds': unlearning_seconds, 'minutes': unlearning_seconds / 60},
])
runtime_comparison['speedup_vs_full_retraining'] = full_retraining_seconds / runtime_comparison['seconds']
runtime_comparison.to_csv(PATHS['results'] / 'final_profile_runtime_comparison.csv', index=False)
display(runtime_comparison)

In [ ]:
expected = [
    PATHS['results'] / 'final_profile_recall_comparison.csv',
    PATHS['results'] / 'final_profile_membership_comparison.csv',
    PATHS['results'] / 'final_profile_runtime_comparison.csv',
]
assert all(path.exists() for path in expected)
print('Profile-memory unlearning comparison complete.')